<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase27_validacion_reglas_riesgo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clase 27: validación hacia adelante de las reglas de riesgo

Se prueba si elegir con el pasado los parámetros de las reglas de riesgo del carry funciona fuera de muestra, contra la familia completa de reglas y contra el azar.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm

# EJEMPLO A MANO 1: elegir con el pasado, tres estrategias y dos periodos
entrenamiento = {"X": 0.8, "Y": 0.5, "Z": 0.2}        # Sharpe en el periodo de entrenamiento
prueba = {"X": -0.1, "Y": 0.4, "Z": 0.3}              # Sharpe en el periodo siguiente
elegida = max(entrenamiento, key=entrenamiento.get)
orden = sorted(prueba, key=prueba.get, reverse=True)
print("Elegida con el pasado:", elegida, "| su Sharpe despues: %.1f | su lugar entre las 3: %d" % (prueba[elegida], orden.index(elegida) + 1))
print("Promedio de las 3 despues: %.2f" % np.mean(list(prueba.values())))

# EJEMPLO A MANO 2: el mejor de muchas estrategias SIN ventaja real
# 19 estrategias con Sharpe verdadero de 0 y 4.23 años de datos: su Sharpe medido tiene un error estandar de 0.49
rng = np.random.default_rng(1)
se, n_estrategias = 1 / np.sqrt(4.23), 19
mejores = [rng.normal(0, se, n_estrategias).max() for _ in range(5000)]
print("\nSharpe del MEJOR de %d estrategias sin ventaja: promedio %.2f | 95%% de las veces menor a %.2f" % (
    n_estrategias, np.mean(mejores), np.percentile(mejores, 95)))


Elegida con el pasado: X | su Sharpe despues: -0.1 | su lugar entre las 3: 3
Promedio de las 3 despues: 0.20

Sharpe del MEJOR de 19 estrategias sin ventaja: promedio 0.89 | 95% de las veces menor a 1.36


In [2]:
# Funciones (las mismas de la clase 25, mas medidas de cola)
def vol_ewma(rend, lam=0.94):
    """Volatilidad anualizada con EWMA, usando datos hasta t (inclusive)."""
    return np.sqrt((rend ** 2).ewm(alpha=1 - lam, adjust=False).mean() * 252)

def backtest(rend, pos, rezago=1, costo_bps=0.0):
    """La posicion al cierre de t gana el rendimiento de t+1 (rezago=1). 'rend' es el rendimiento de estar largo USD."""
    pos_usada = pos.shift(rezago)
    bruto = (pos_usada * rend).mean(axis=1)
    rotacion = pos_usada.diff().abs().mean(axis=1).fillna(0.0)
    return bruto - rotacion * costo_bps / 10000, rotacion

def sharpe(x):
    x = x.dropna()
    return x.mean() / x.std() * np.sqrt(252)

def metricas(pnl):
    pnl = pnl.dropna()
    anios = len(pnl) / 252
    s = sharpe(pnl)
    t = s * np.sqrt(anios)
    return {"rend_%": pnl.mean() * 252 * 100, "vol_%": pnl.std() * np.sqrt(252) * 100, "Sharpe": s,
            "t": t, "p": 2 * (1 - norm.cdf(abs(t)))}

def caida_maxima(pnl):
    c = pnl.dropna().cumsum()
    return (c - c.cummax()).min() * 100


def cola(pnl):
    pnl = pnl.dropna()
    mensual = pnl.resample("ME").sum() * 100
    peores = pnl.sort_values().iloc[: max(1, int(len(pnl) * 0.05))]
    return {"peor_mes_%": mensual.min(), "asimetria": mensual.skew(), "caida_max_%": caida_maxima(pnl),
            "media_peor_5%_dia_%": peores.mean() * 100}


In [3]:
# Datos (igual que la clase 25): precios de seis pares contra el dolar y tasas de interes de corto plazo (FRED, mensuales)
pares = {"USD/MXN": ("MXN=X", False, "MXN"), "USD/JPY": ("JPY=X", False, "JPY"), "USD/EUR": ("EURUSD=X", True, "EUR"),
         "USD/GBP": ("GBPUSD=X", True, "GBP"), "USD/CAD": ("CAD=X", False, "CAD"), "USD/BRL": ("BRL=X", False, "BRL")}
px = {}
for nombre, (ticker, invertir, _) in pares.items():
    p = yf.download(ticker, start="2020-10-07", end="2026-10-22", progress=False, auto_adjust=True)["Close"].squeeze()
    px[nombre] = 1 / p if invertir else p
rend = pd.DataFrame(px).dropna().pct_change().dropna()     # rendimiento de estar largo USD (precio en moneda extranjera por dolar)
CALENTAMIENTO = 505                                        # las mismas fechas de evaluacion de la clase 24
print("Dias:", len(rend), "| del", rend.index[0].date(), "al", rend.index[-1].date())

# Tasas de corto plazo de FRED (una por mes). Para cada moneda se prueban varias series, en este orden, y se usa la primera que baje y este al dia.
ids_tasa = {"USD": ["IR3TIB01USM156N", "IRSTCI01USM156N", "FEDFUNDS"],
            "MXN": ["IR3TIB01MXM156N", "IRSTCI01MXM156N", "INTDSRMXM193N"],
            "JPY": ["IR3TIB01JPM156N", "IRSTCI01JPM156N", "INTDSRJPM193N"],
            "EUR": ["IR3TIB01EZM156N", "IRSTCI01EZM156N", "ECBDFR"],
            "GBP": ["IR3TIB01GBM156N", "IRSTCI01GBM156N", "INTDSRGBM193N"],
            "CAD": ["IR3TIB01CAM156N", "IRSTCI01CAM156N", "INTDSRCAM193N"],
            "BRL": ["IRSTCI01BRM156N", "INTDSRBRM193N", "IR3TIB01BRM156N"]}
def tasa_fred(serie):
    s = pd.read_csv(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={serie}", index_col=0, parse_dates=True).iloc[:, 0]
    s = pd.to_numeric(s, errors="coerce").dropna()
    return s.resample("MS").mean().dropna()                # una observacion por mes (promedio), tambien para series diarias como ECBDFR
limite = rend.index[-1] - pd.DateOffset(months=6)          # una serie que termina antes de esta fecha se considera desactualizada
tasas, usada = {}, {}
for moneda, lista in ids_tasa.items():
    bajadas = {}
    for sid in lista:
        try:
            bajadas[sid] = tasa_fred(sid)
        except Exception as e:
            print("   no sirvio %s (%s): %s" % (moneda, sid, str(e)[:60]))
    if not bajadas:
        continue
    al_dia = [sid for sid in bajadas if bajadas[sid].index[-1] >= limite]
    elegida = al_dia[0] if al_dia else max(bajadas, key=lambda k: bajadas[k].index[-1])
    tasas[moneda], usada[moneda] = bajadas[elegida], elegida
    if not al_dia:
        print("   AVISO: ninguna serie de %s esta al dia; se usa %s" % (moneda, elegida))
faltan = [m for m in ids_tasa if m not in tasas]
if faltan:
    print("SIN TASA para:", faltan, "-> se quitan esos pares del analisis (ya no son seis, y eso cambia la comparacion con la clase 24)")
    pares = {k: v for k, v in pares.items() if v[2] in tasas}
    rend = rend[list(pares)]
if "USD" not in tasas:
    raise SystemExit("Sin tasa del dolar. Mandame este mensaje.")
for m, s in tasas.items():
    print("%s (%s): ultima fecha %s | ultimo valor %.2f%% | promedio desde 2020-10: %.2f%%" % (m, usada[m], s.index[-1].date(), s.iloc[-1], s.loc["2020-10-01":].mean()))

# Sin mirar el futuro: la tasa del mes m (publicada despues) solo se usa a partir del mes m+1
tasas_df = pd.DataFrame(tasas)
tasas_df.index = tasas_df.index + pd.DateOffset(months=1)
tasas_d = tasas_df.reindex(tasas_df.index.union(rend.index)).ffill().reindex(rend.index)
for m, s in tasas.items():
    if s.index[-1] + pd.DateOffset(months=1) < rend.index[-1]:
        print("AVISO: la tasa de %s se congela desde %s (ultimo dato de FRED)" % (m, s.index[-1].date()))

# Diferencial anual: tasa del dolar menos tasa de la otra moneda (positivo: estar largo USD cobra intereses)
dif = pd.DataFrame({p: (tasas_d["USD"] - tasas_d[c]) / 100 for p, (_, _, c) in pares.items()})
rend_int = dif / 252                                       # intereses diarios de estar largo USD
rend_tot = rend + rend_int                                 # rendimiento total de estar largo USD
print("\nDiferencial promedio (USD menos la otra moneda, en %), del 2022-09-15 al final:")
print((dif.iloc[CALENTAMIENTO:].mean() * 100).round(2).to_string())


Dias: 1570 | del 2020-10-08 al 2026-10-21
   no sirvio BRL (IR3TIB01BRM156N): HTTP Error 404: Not Found
USD (IR3TIB01USM156N): ultima fecha 2026-10-01 | ultimo valor 4.12% | promedio desde 2020-10: 3.28%
MXN (IR3TIB01MXM156N): ultima fecha 2026-10-01 | ultimo valor 6.81% | promedio desde 2020-10: 8.43%
JPY (IR3TIB01JPM156N): ultima fecha 2026-09-01 | ultimo valor 1.68% | promedio desde 2020-10: 0.32%
EUR (ECBDFR): ultima fecha 2026-11-01 | ultimo valor 2.50% | promedio desde 2020-10: 1.74%
GBP (IRSTCI01GBM156N): ultima fecha 2026-10-01 | ultimo valor 3.73% | promedio desde 2020-10: 3.03%
CAD (IR3TIB01CAM156N): ultima fecha 2026-10-01 | ultimo valor 2.45% | promedio desde 2020-10: 2.61%
BRL (IRSTCI01BRM156N): ultima fecha 2026-10-01 | ultimo valor 13.75% | promedio desde 2020-10: 11.22%
AVISO: la tasa de JPY se congela desde 2026-09-01 (ultimo dato de FRED)

Diferencial promedio (USD menos la otra moneda, en %), del 2022-09-15 al final:
USD/MXN   -5.30
USD/JPY    4.05
USD/EUR    1.80
US

In [4]:
# LA FAMILIA DE REGLAS DE RIESGO SOBRE EL CARRY (19 estrategias, todas con rendimiento total y rezago de un dia)
escala = (0.10 / vol_ewma(rend)).clip(upper=3.0)
carry = np.sign(dif)
pos_A = carry * escala
pnl_A = backtest(rend_tot, pos_A)[0]
OBJETIVO = 0.045
cum = pnl_A.fillna(0).cumsum()
caida = cum - cum.cummax()

familia = {"A: sin regla": pnl_A}
for lam in (0.90, 0.94, 0.97, 0.99):                                   # B: volatilidad objetivo
    vol_c = np.sqrt((pnl_A ** 2).ewm(alpha=1 - lam, adjust=False).mean() * 252)
    for tope in (1.5, 2.0, 3.0):
        lev = (OBJETIVO / vol_c).clip(upper=tope).fillna(1.0)
        familia[f"B: lambda {lam}, tope {tope}"] = backtest(rend_tot, pos_A.mul(lev, axis=0))[0]
for umbral in (0.02, 0.03, 0.05):                                      # C: reducir tras una caida
    for expo in (0.25, 0.5):
        lev = pd.Series(np.where(caida < -umbral, expo, 1.0), index=caida.index)
        familia[f"C: caida {int(umbral*100)}%, expo {expo}"] = backtest(rend_tot, pos_A.mul(lev, axis=0))[0]
P = pd.DataFrame(familia).iloc[CALENTAMIENTO:]
print("Estrategias en la familia:", P.shape[1], "| dias:", len(P))

# Sharpe de cada una en TODO el periodo (ojo: elegir la mejor de aqui seria usar el futuro)
sh = P.apply(sharpe)
print("\nSharpe de toda la familia (de menor a mayor):")
print(sh.sort_values().round(2).to_string())
mejor = sh.idxmax()
t_mejor = sh.max() * np.sqrt(len(P) / 252)
p_mejor = 2 * (1 - norm.cdf(t_mejor))
print("\nLa mejor en toda la muestra: %s | Sharpe %.2f | p %.3f | p con Bonferroni (%d pruebas): %.3f" % (
    mejor, sh.max(), p_mejor, P.shape[1], min(1.0, p_mejor * P.shape[1])))


Estrategias en la familia: 19 | dias: 1065

Sharpe de toda la familia (de menor a mayor):
C: caida 2%, expo 0.25      0.75
C: caida 3%, expo 0.25      0.84
C: caida 5%, expo 0.25      0.84
C: caida 2%, expo 0.5       0.85
C: caida 5%, expo 0.5       0.88
C: caida 3%, expo 0.5       0.89
A: sin regla                0.94
B: lambda 0.99, tope 2.0    1.01
B: lambda 0.99, tope 3.0    1.01
B: lambda 0.99, tope 1.5    1.01
B: lambda 0.97, tope 3.0    1.06
B: lambda 0.97, tope 2.0    1.06
B: lambda 0.97, tope 1.5    1.06
B: lambda 0.94, tope 3.0    1.11
B: lambda 0.94, tope 2.0    1.11
B: lambda 0.94, tope 1.5    1.12
B: lambda 0.9, tope 3.0     1.14
B: lambda 0.9, tope 2.0     1.14
B: lambda 0.9, tope 1.5     1.18

La mejor en toda la muestra: B: lambda 0.9, tope 1.5 | Sharpe 1.18 | p 0.016 | p con Bonferroni (19 pruebas): 0.295


In [5]:
# VALIDACION HACIA ADELANTE: cada 126 dias se elige la estrategia con mejor Sharpe en el pasado (ventana creciente, minimo 252 dias) y se usa los 126 dias siguientes
paso, minimo = 126, 252
filas, oos = [], []
for ini in range(minimo, len(P) - 1, paso):
    entreno, prueba = P.iloc[:ini], P.iloc[ini: ini + paso]
    elegida = entreno.apply(sharpe).idxmax()
    sh_prueba = prueba.apply(sharpe)
    lugar = sh_prueba.rank(ascending=False)[elegida]                   # 1 = la mejor de las 19 en la prueba
    filas.append({"desde": prueba.index[0].date(), "elegida": elegida, "Sharpe entreno": sharpe(entreno[elegida]),
                  "Sharpe prueba": sharpe(prueba[elegida]), "lugar entre 19": int(lugar),
                  "Sharpe prueba de A": sharpe(prueba["A: sin regla"]), "Sharpe prueba del promedio": sharpe(prueba.mean(axis=1))})
    oos.append(prueba[elegida])
tabla = pd.DataFrame(filas)
print(tabla.round(2).to_string(index=False))
oos = pd.concat(oos)
print("\nLugar promedio de la elegida en la prueba: %.1f de 19 (sin habilidad se espera 10)" % tabla["lugar entre 19"].mean())


     desde                 elegida  Sharpe entreno  Sharpe prueba  lugar entre 19  Sharpe prueba de A  Sharpe prueba del promedio
2023-09-04 B: lambda 0.9, tope 1.5            1.44           1.33               1                0.98                        1.07
2024-02-27 B: lambda 0.9, tope 1.5            1.41          -1.18              18               -0.78                       -0.85
2024-08-21 B: lambda 0.9, tope 1.5            0.73           0.82               1                0.42                        0.62
2025-02-14 B: lambda 0.9, tope 1.5            0.74           0.39              11                0.73                        0.42
2025-08-13 B: lambda 0.9, tope 1.5            0.68           2.96               2                2.83                        2.87
2026-02-09 B: lambda 0.9, tope 1.5            1.00           2.24               1                1.85                        2.00
2026-08-04 B: lambda 0.9, tope 1.5            1.15           1.58               2         

In [6]:
# RESULTADO FUERA DE MUESTRA (mismas fechas para todas)
fechas = oos.index
comp = {"Walk forward (elegir con el pasado)": oos,
        "A: sin regla": P.loc[fechas, "A: sin regla"],
        "Promedio de las 19": P.loc[fechas].mean(axis=1),
        "B: lambda 0.94, tope 2.0 (la de la clase 26)": P.loc[fechas, "B: lambda 0.94, tope 2.0"],
        "Mejor en toda la muestra (usa el futuro)": P.loc[fechas, mejor]}
res = pd.DataFrame({k: {**metricas(v), "caida_max_%": caida_maxima(v)} for k, v in comp.items()}).T
print(res.round(3).to_string())
print("\nDias fuera de muestra: %d (%.2f años) | error estandar aproximado del Sharpe: %.2f" % (len(fechas), len(fechas) / 252, 1 / np.sqrt(len(fechas) / 252)))
print("Veces que se cambio de estrategia entre periodos:", int((tabla["elegida"] != tabla["elegida"].shift()).sum() - 1))


                                              rend_%  vol_%  Sharpe      t      p  caida_max_%
Walk forward (elegir con el pasado)            5.279  4.828   1.093  1.964  0.050       -7.372
A: sin regla                                   4.268  4.605   0.927  1.665  0.096       -7.435
Promedio de las 19                             4.441  4.381   1.014  1.821  0.069       -6.396
B: lambda 0.94, tope 2.0 (la de la clase 26)   5.037  4.759   1.058  1.901  0.057       -6.874
Mejor en toda la muestra (usa el futuro)       5.279  4.828   1.093  1.964  0.050       -7.372

Dias fuera de muestra: 813 (3.23 años) | error estandar aproximado del Sharpe: 0.56
Veces que se cambio de estrategia entre periodos: 0


In [7]:
# Rotacion de cuatro reglas de la familia, como referencia para los costos (no se simulan costos sobre la elegida porque cambia de regla en cada periodo)
rot = {}
for nombre in ("A: sin regla", "B: lambda 0.94, tope 2.0", "B: lambda 0.90, tope 3.0", "C: caida 3%, expo 0.5"):
    if nombre.startswith("A"): pos = pos_A
    elif nombre.startswith("B"):
        lam, tope = float(nombre.split("lambda ")[1].split(",")[0]), float(nombre.split("tope ")[1])
        vol_c = np.sqrt((pnl_A ** 2).ewm(alpha=1 - lam, adjust=False).mean() * 252)
        pos = pos_A.mul((OBJETIVO / vol_c).clip(upper=tope).fillna(1.0), axis=0)
    else:
        pos = pos_A.mul(pd.Series(np.where(caida < -0.03, 0.5, 1.0), index=caida.index), axis=0)
    rot[nombre] = backtest(rend_tot, pos)[1].iloc[CALENTAMIENTO:].mean() * 252
print("Rotacion anual (unidades operadas por año):")
print(pd.Series(rot).round(2).to_string())


Rotacion anual (unidades operadas por año):
A: sin regla                11.49
B: lambda 0.94, tope 2.0    19.94
B: lambda 0.90, tope 3.0    27.25
C: caida 3%, expo 0.5       13.92
